# RAG-based Fake News Detector
**Stack:** `sentence-transformers` · `FAISS` · `Anthropic Claude` · `Streamlit`

**Pipeline:**
```
User claim → Sentence Embedding (all-MiniLM-L6-v2) → FAISS Cosine Search
           → Top-K Evidence Retrieval → LLM (Claude) + Context → Verdict
```
**Author:** Srimath Koilkandadai Srihita

In [19]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq
import json

print('All imports successful ✓')

All imports successful ✓


## 1. Knowledge Base
In production, this would be loaded from PDFs/news articles using LangChain document loaders.

In [2]:
KNOWLEDGE_BASE = [
    {"text": "WHO has not endorsed any single food item as a cure for cancer. Cancer treatment requires medical intervention.", "source": "WHO Health Bulletin, 2023"},
    {"text": "Coffee consumption has been linked to reduced risk of certain cancers in observational studies, but causation is not established.", "source": "NIH Cancer Research Journal, 2022"},
    {"text": "Climate change is real and primarily driven by human activity, confirmed by 97% of climate scientists.", "source": "IPCC Sixth Assessment Report, 2021"},
    {"text": "Vaccines do not cause autism. The original 1998 Wakefield study was retracted for fraud and ethical violations.", "source": "Lancet retraction 2010; CDC 2023"},
    {"text": "The 2020 US presidential election was not stolen. Over 60 courts dismissed election fraud claims for lack of evidence.", "source": "AP Fact Check, Reuters 2021"},
    {"text": "5G technology does not spread COVID-19. Viruses cannot travel on radio waves or mobile networks.", "source": "Full Fact, WHO 2020"},
    {"text": "India's GDP grew by approximately 8.2% in FY2023-24, making it one of the fastest growing major economies.", "source": "Ministry of Statistics India, 2024"},
    {"text": "Drinking bleach or disinfectants is extremely dangerous and does not cure or prevent COVID-19.", "source": "CDC Emergency Alert, 2020"},
    {"text": "The Earth is approximately 4.5 billion years old, based on radiometric dating of rocks and meteorites.", "source": "USGS Earth Science, 2023"},
    {"text": "Ivermectin has not been proven effective against COVID-19 in large randomized controlled trials.", "source": "NIH COVID Treatment Guidelines, 2022"},
    {"text": "The Great Wall of China is not visible from space with the naked eye — a popular myth contradicted by astronauts.", "source": "NASA Space Facts, 2021"},
    {"text": "ChatGPT was launched by OpenAI in November 2022 and reached 100 million users in two months.", "source": "OpenAI Blog, Reuters 2023"},
]

print(f'Knowledge base loaded: {len(KNOWLEDGE_BASE)} documents ✓')

Knowledge base loaded: 12 documents ✓


## 2. Embed Knowledge Base with Sentence Transformers

In [3]:
# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded ✓')

# Encode all knowledge base documents
texts = [doc['text'] for doc in KNOWLEDGE_BASE]
embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

print(f'Embedding shape: {embeddings.shape}')  # (n_docs, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded ✓


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (12, 384)


## 3. Build FAISS Index

In [4]:
# L2-normalize for cosine similarity via inner product
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]  # 384 for all-MiniLM-L6-v2
index = faiss.IndexFlatIP(dim)  # Inner Product on normalized = Cosine
index.add(embeddings)

print(f'FAISS index built: {index.ntotal} vectors, dim={dim} ✓')

FAISS index built: 12 vectors, dim=384 ✓


## 4. Retrieval Function

In [7]:
def retrieve(query: str, k: int = 1) -> list:
    """Embed query and find top-k most similar knowledge base documents."""
    q_vec = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        doc = KNOWLEDGE_BASE[idx].copy()
        doc['similarity'] = round(float(score), 4)
        results.append(doc)
    return results

# Test retrieval
test_docs = retrieve('Does coffee cure cancer?', k=1)
print('Top-3 retrieved documents:')
for i, doc in enumerate(test_docs):
    print(f"  [{i+1}] similarity={doc['similarity']:.4f} | {doc['text'][:80]}...")

Top-3 retrieved documents:
  [1] similarity=0.7560 | Coffee consumption has been linked to reduced risk of certain cancers in observa...


## 5. LLM Classification with Retrieved Context

In [40]:
import os
import json
from groq import Groq

API_KEY = "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"

def classify_claim(query: str, k: int = 3):
    """Full RAG pipeline: retrieve → augment → generate."""

    # Step 1: Retrieve evidence
    evidence = retrieve(query, k=k)

    # Step 2: Build context
    context = "\n".join(
        [
            f"[Evidence {i+1}] {e['text']} (Source: {e['source']})"
            for i, e in enumerate(evidence)
        ]
    )

    system_prompt = """
You are a fact-checking assistant using Retrieval-Augmented Generation (RAG).

You receive a news claim and retrieved evidence from a trusted knowledge base.

Classify the claim as:
REAL
FAKE
UNCERTAIN

Return ONLY valid JSON:

{
  "verdict": "REAL",
  "confidence": "High",
  "explanation": "2-3 sentences citing the evidence"
}
"""

    user_prompt = f"""
News claim:
{query}

Retrieved evidence:
{context}

Classify the claim based on this evidence.
"""

    # Step 3: Call Groq
    client = Groq(api_key=API_KEY)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )

    # Step 4: Extract response
    raw = response.choices[0].message.content.strip()

    # Step 5: Parse JSON safely
    try:
        result = json.loads(raw)
    except:
        result = {
            "verdict": "UNCERTAIN",
            "confidence": "Low",
            "explanation": raw
        }

    result["retrieved_evidence"] = evidence

    return result

print("classify_claim() function defined ✓")

classify_claim() function defined ✓


## 6. Run the Full Pipeline

In [41]:
claim = "Vaccines do not cause autism."

result = classify_claim(claim)

print("Verdict:", result["verdict"])
print("Confidence:", result["confidence"])
print("Explanation:", result["explanation"])

print("\nRetrieved Evidence:")
for e in result["retrieved_evidence"]:
    print("-", e["text"])

Verdict: REAL
Confidence: High
Explanation: The claim that vaccines do not cause autism is supported by Evidence 1, which states that the original 1998 Wakefield study was retracted for fraud and ethical violations, and is further confirmed by reputable sources such as the CDC in 2023. Evidence 1 directly addresses the claim, providing a clear and reliable source to back it up. The other evidence pieces, while relevant to other health topics, do not contradict this claim.

Retrieved Evidence:
- Vaccines do not cause autism. The original 1998 Wakefield study was retracted for fraud and ethical violations.
- WHO has not endorsed any single food item as a cure for cancer. Cancer treatment requires medical intervention.
- Ivermectin has not been proven effective against COVID-19 in large randomized controlled trials.


## 7. Architecture Notes

### Why RAG over pure LLM?
- LLMs hallucinate facts. RAG grounds the response in retrieved, trusted documents
- Knowledge base can be updated without retraining the model
- Retrieval provides explainability — you can show *which* evidence influenced the verdict

### Why FAISS?
- Approximate Nearest Neighbor (ANN) search — O(log n) instead of O(n) brute force
- `IndexFlatIP` = exact inner product (cosine on normalized vectors)
- For larger corpora: `IndexIVFFlat` (inverted file, faster) or `IndexHNSW` (graph-based)

### Why `all-MiniLM-L6-v2`?
- 384-dim embeddings, very fast (6-layer MiniLM)
- Trained on 1B sentence pairs for semantic similarity
- Good balance of speed vs quality. For production: `all-mpnet-base-v2` (768-dim)

### Production extensions
- Replace static KB with LangChain document loaders (PDFs, web scraping)
- Add `RecursiveCharacterTextSplitter` for chunking long documents
- Use Pinecone or Chroma for persistent vector storage
- Add re-ranking step (cross-encoder) for better precision
- Fine-tune embeddings on domain-specific data